# Lab 4.1 — Prompt Templates, LangChain Hub and Few-Shot Prompting with Ollama
**Course:** AI Agents · **Unit:** UC4 — AIAgents · **Platform:** Google Colab (T4 GPU recommended)

An agent's "brain" is an LLM, and the only way we talk to it is through **prompts**. Up to now we wrote prompts as plain strings. In this lab we move to **LangChain** with a **local open model served by Ollama**, and we start treating prompts as reusable, inspectable and testable **software components**.

### Learning objectives
1. Run an open LLM locally with **Ollama** and call it from LangChain (`ChatOllama`).
2. Build reusable prompts with `PromptTemplate` and `ChatPromptTemplate` (roles, variables, `partial`, `MessagesPlaceholder`).
3. Compose prompt → model → parser with the LCEL pipe operator `|`, and run it with `invoke`, `batch` and `stream`.
4. Pull, inspect and adapt community prompts from the **LangChain Hub**, and use a *prompt-maker* prompt to improve a weak prompt.
5. **Measure** the effect of **few-shot prompting** against a zero-shot baseline, with fixed and with semantically selected examples.

### Lab structure
| Part | Topic | What you do |
|---|---|---|
| 0 | Setup: Ollama + LangChain, the LCEL pattern | Run |
| 1 | Prompt templates | Run, then **Exercise 1** |
| 2 | Improving prompts with the LangChain Hub | Run, then **Exercise 2** |
| 3 | Few-shot prompting | Run, then **Exercise 3** |
| 4 | Questions | Answer |

### Running this notebook in Google Colab
1. **File → Upload notebook** and pick this `.ipynb` file.
2. Turn on the GPU: **Runtime → Change runtime type → T4 GPU → Save** (CPU also works, but every answer is slower).
3. Run the cells **in order** (`Shift + Enter`). Part 0 downloads Ollama and the models (≈ 2–3 minutes the first time).

> **Running on your own laptop instead?** Install Ollama from <https://ollama.com/download> (it runs as a background service), skip cell 0.1, and run the rest unchanged. The same code works in both places — that is one of the advantages of a local model server.

## Part 0 — Setup

### What is Ollama?
[Ollama](https://ollama.com) is a small server that downloads **open-weight models** (Llama, Qwen, Gemma, Mistral…) in a compressed (*quantized*) format and serves them through a local REST API at `http://localhost:11434`. Compared with calling a commercial API:

| | Local model (Ollama) | Commercial API |
|---|---|---|
| Cost per token | none | pay per token |
| API key | not needed | needed |
| Data privacy | text never leaves the machine | text is sent to the provider |
| Quality | smaller models, weaker reasoning | state-of-the-art models |

The package `langchain-ollama` wraps that API in a LangChain chat model, so everything you learn here also works if you later swap `ChatOllama` for another provider.

In [7]:
#%% 0.1 Install Ollama and start the server (once per Colab session, ~1 minute)
import os, subprocess, time, urllib.request

!sudo apt-get -qq update > /dev/null 2>&1 && sudo apt-get -qq install -y zstd pciutils > /dev/null 2>&1
!curl -fsSL https://ollama.com/install.sh | sh > /dev/null 2>&1

os.environ["OLLAMA_HOST"] = "127.0.0.1:11434"
os.environ["OLLAMA_NUM_PARALLEL"] = "4"      # let the server answer several requests at once (used by .batch())
ollama_log = open("ollama.log", "w")
ollama_proc = subprocess.Popen(["ollama", "serve"], stdout=ollama_log, stderr=subprocess.STDOUT)

def wait_for_ollama(timeout=60):
    # Poll the local API until the server answers, or give up after `timeout` seconds.
    for _ in range(timeout):
        try:
            urllib.request.urlopen("http://127.0.0.1:11434", timeout=2)
            return True
        except Exception:
            time.sleep(1)
    return False

if wait_for_ollama():
    print("✅ Ollama server is running")
    !ollama --version
else:
    print("❌ The server did not start. Last lines of the log:")
    print(open("ollama.log").read()[-2000:])

✅ Ollama server is running
ollama version is 0.40.16


### 0.2 Download the models
- **Chat model:** `llama3.2:3b` (≈ 2 GB, 3 billion parameters). If the download is slow, use `qwen2.5:1.5b` (≈ 1 GB) — every cell still works, but expect more mistakes.
- **Embedding model:** `nomic-embed-text` (≈ 270 MB). It turns text into vectors; we only need it in Part 3.5.

In [8]:
#%% 0.2 Download the models (first time only)
MODEL = "llama3.2:3b"            # faster alternative: "qwen2.5:1.5b"
EMBED_MODEL = "nomic-embed-text"

!ollama pull {MODEL} > /dev/null 2>&1
!ollama pull {EMBED_MODEL} > /dev/null 2>&1
!ollama list

NAME                       ID              SIZE      MODIFIED               
nomic-embed-text:latest    0a109f422b47    274 MB    Less than a second ago    
llama3.2:3b                a80c4f17acd5    2.0 GB    Less than a second ago    


In [9]:
#%% 0.3 Install the LangChain packages we need
%pip install -q -U langchain-core langchain-ollama langsmith

In [10]:
#%% 0.4 Check versions
from importlib.metadata import version
for pkg in ("langchain-core", "langchain-ollama", "langsmith"):
    print(f"{pkg:17s}", version(pkg))

langchain-core    1.8.0
langchain-ollama  1.1.0
langsmith         0.14.17


### 0.5 First call: the model as a LangChain object
`ChatOllama` exposes Ollama's generation options with their Ollama names. The ones we use:

| Option | Meaning | Lab 2.5 equivalent |
|---|---|---|
| `temperature` | sharpens / flattens the next-token distribution | temperature |
| `num_predict` | maximum number of tokens to generate | `max_new_tokens` |
| `num_ctx` | context window: how many tokens of prompt + answer the model can see | – |

> ⚠️ Ollama's default `num_ctx` is small (a few thousand tokens) and **longer prompts are silently truncated**. Few-shot prompts grow quickly, so we set it explicitly.

We use `temperature=0` in almost every cell: we want differences between outputs to come from the **prompt**, not from sampling.

A chat model returns an **`AIMessage`**: the text is in `.content`, and token counts are in `.usage_metadata`.

In [11]:
#%% 0.5 First call
from langchain_ollama import ChatOllama

llm = ChatOllama(model=MODEL, temperature=0, num_predict=256, num_ctx=4096)
llm_long = ChatOllama(model=MODEL, temperature=0, num_predict=700, num_ctx=4096)   # for longer answers

response = llm.invoke("In one sentence, what is a prompt template?")
print(type(response).__name__)
print(response.content)
print("tokens:", response.usage_metadata)

AIMessage
A prompt template is a pre-formatted structure or framework used to guide the creation of text, such as a story, article, or conversation, by providing a starting point for the user to fill in with their own ideas, words, and tone.
tokens: {'input_tokens': 35, 'output_tokens': 51, 'total_tokens': 86}


### 0.6 The LCEL pattern: `prompt | llm | parser`
In LangChain every component (prompt, model, parser…) is a **Runnable** with the same interface:

| Method | Input | Output |
|---|---|---|
| `.invoke(x)` | one input | one output |
| `.batch([x1, x2, …])` | a list of inputs | a list of outputs (sent in parallel) |
| `.stream(x)` | one input | the output in chunks, as it is generated |

The pipe operator `a | b` (LangChain Expression Language, **LCEL**) builds a new Runnable that feeds the output of `a` into `b`. The most common pipeline is:

```
dict of variables ──► prompt ──► PromptValue ──► llm ──► AIMessage ──► StrOutputParser ──► str
```

This lab uses this one pattern everywhere; the next labs (chains) are built entirely out of it.

In [12]:
#%% 0.6 invoke, batch and stream
import time
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import StrOutputParser

prompt = ChatPromptTemplate.from_template("Explain {concept} to a first-year student in two sentences.")
chain = prompt | llm | StrOutputParser()

print("invoke →", chain.invoke({"concept": "recursion"}), "\n")

concepts = ["a variable", "a loop", "a function", "a list"]
t0 = time.time(); one_by_one = [chain.invoke({"concept": c}) for c in concepts]; t_loop = time.time() - t0
t0 = time.time(); batched = chain.batch([{"concept": c} for c in concepts]); t_batch = time.time() - t0
print(f"4 calls one by one: {t_loop:.1f} s | one .batch() call: {t_batch:.1f} s")

print("\nstream → ", end="")
for chunk in chain.stream({"concept": "an algorithm"}):
    print(chunk, end="", flush=True)

invoke → Recursion is a programming concept where a function calls itself repeatedly until it reaches a base case that stops the recursion, allowing the function to solve a problem by breaking it down into smaller instances of the same problem. Think of it like a set of Russian nesting dolls - each doll is a smaller version of the original, and the function "unfolds" the problem by solving each smaller instance until it reaches the simplest case. 

4 calls one by one: 4.0 s | one .batch() call: 2.1 s

stream → An algorithm is like a recipe for a computer to follow, where it takes in some input, does a series of steps, and produces an output. Think of it like a step-by-step guide that the computer can understand, like "take this number, add 2, multiply by 3, and then print the result".

---
## Part 1 — Prompt templates

Writing prompts as f-strings (`f"Translate {text} to {lang}"`) works for one call but breaks down quickly:
- nothing checks that every variable was filled;
- you cannot inspect or reuse the prompt independently of the code that sends it;
- instructions and user data are mixed in one string;
- there are no **roles** (system / human / AI), which chat models are trained to use.

A **prompt template** is text with `{placeholders}` plus the list of its **input variables**. Filling it produces a **`PromptValue`**, which can be turned into a plain string (for completion models) or into a list of chat messages (for chat models).

In [13]:
#%% 1.1 PromptTemplate: a string with variables
from langchain_core.prompts import PromptTemplate

translate = PromptTemplate.from_template(
    "Translate the following text from {source} to {target}. Return only the translation.\n\nText: {text}")
print("input variables:", translate.input_variables)

value = translate.invoke({"source": "English", "target": "Spanish", "text": "Agents perceive and act."})
print(type(value).__name__, "→", repr(value.to_string()))
print("as chat messages:", value.to_messages())

try:
    translate.invoke({"source": "English", "text": "Hello"})      # 'target' is missing
except Exception as e:
    print(f"\n❌ Caught before calling the model → {type(e).__name__}: {e}")

input variables: ['source', 'target', 'text']
StringPromptValue → 'Translate the following text from English to Spanish. Return only the translation.\n\nText: Agents perceive and act.'
as chat messages: [HumanMessage(content='Translate the following text from English to Spanish. Return only the translation.\n\nText: Agents perceive and act.', additional_kwargs={}, response_metadata={})]

❌ Caught before calling the model → KeyError: "Input to PromptTemplate is missing variables {'target'}.  Expected: ['source', 'target', 'text'] Received: ['source', 'text']\nNote: if you intended {target} to be part of the string and not a variable, please escape it with double curly braces like: '{{target}}'.\nFor troubleshooting, visit: https://docs.langchain.com/oss/python/langchain/errors/INVALID_PROMPT_INPUT "


### 1.2 `ChatPromptTemplate`: messages with roles
Chat models are trained on conversations made of **messages with roles**:

| Role | Tuple shorthand | Used for |
|---|---|---|
| System | `("system", ...)` | persona, rules, output format — applies to the whole conversation |
| Human | `("human", ...)` | the user's request (the data) |
| AI | `("ai", ...)` | previous answers of the model — also used to write **examples** (Part 3) |

Each message can contain variables. `.pretty_print()` shows the messages exactly as the model will receive them.

In [14]:
#%% 1.2 A tutor prompt with a system and a human message
tutor_prompt = ChatPromptTemplate.from_messages([
    ("system", "You are a patient programming tutor. Explain at a {level} level, "
               "answer in {language}, and use at most {max_sentences} sentences."),
    ("human", "Explain this concept: {topic}"),
])
print("input variables:", tutor_prompt.input_variables)

messages = tutor_prompt.invoke({"level": "beginner", "language": "English",
                                "max_sentences": 3, "topic": "recursion"}).to_messages()
for m in messages:
    m.pretty_print()

input variables: ['language', 'level', 'max_sentences', 'topic']
================================ System Message ================================

You are a patient programming tutor. Explain at a beginner level, answer in English, and use at most 3 sentences.
================================ Human Message =================================

Explain this concept: recursion


### 1.3 One template, many requests
The template is written once and filled many times. With `.batch()` all the requests are sent together.

👉 **Try it:** add a request with `level="expert"` in `"Kichwa"` or another language. Does the 3B model follow every instruction?

In [15]:
#%% 1.3 One template, many requests
tutor_chain = tutor_prompt | llm | StrOutputParser()

requests = [
    {"level": "beginner", "language": "English", "max_sentences": 2, "topic": "recursion"},
    {"level": "expert",   "language": "English", "max_sentences": 2, "topic": "recursion"},
    {"level": "beginner", "language": "Spanish", "max_sentences": 2, "topic": "recursion"},
]
for req, answer in zip(requests, tutor_chain.batch(requests)):
    print(f"━━━ level={req['level']}, language={req['language']} ━━━\n{answer}\n")

━━━ level=beginner, language=English ━━━
Recursion is a programming concept where a function calls itself repeatedly until it reaches a base case that stops the recursion, allowing the function to return a result. Think of it like a set of Russian nesting dolls, where each doll is a smaller version of the same doll, and the function "unfolds" itself until it reaches a simple solution.

━━━ level=expert, language=English ━━━
Recursion is a fundamental concept in programming where a function calls itself repeatedly until a base case is reached, allowing the function to solve a problem by breaking it down into smaller instances of the same problem. This process enables a function to traverse a hierarchical structure, such as a tree or a graph, by recursively exploring its constituent parts.

━━━ level=beginner, language=Spanish ━━━
¡Hola! La recursividad es un patrón de programación donde un algoritmo llama a sí mismo para resolver un problema más pequeño, hasta que se resuelva el problem

###The `.partial()` function in LangChain

The `.partial()` method on a prompt template allows you to **pre-fill** some of its input variables, creating a new prompt template that only requires the remaining variables to be passed at call-time.

This is highly useful in two scenarios:
1. **Partial values:** You already know some parameters (e.g., standard format, database schemas, or deployment environments) but don't have the user request yet.
2. **Partial functions:** You need a value that must be computed dynamically at the exact millisecond the prompt is called (e.g., the current time, today's date, or a session UUID).

In [17]:
# Example 1: Partial with static values
from langchain_core.prompts import PromptTemplate

# Standard translation prompt that expects 3 variables
full_template = PromptTemplate.from_template(
    "Translate this {text} from {source_language} to {target_language}."
)

# We know the deployment is for English to Spanish, so we pre-fill those
spanish_translator = full_template.partial(source_language="English", target_language="Spanish")

# Now, we only need to pass 'text'
print("Input variables needed now:", spanish_translator.input_variables)
prompt_val = spanish_translator.invoke({"text": "Autonomous agents learn from experience."})
print("Generated Prompt:", prompt_val.to_string())

Input variables needed now: ['text']
Generated Prompt: Translate this Autonomous agents learn from experience. from English to Spanish.


### 1.5 Asking for structured output
An agent's code often needs a **data structure**, not prose. Two ingredients:
1. the prompt states the exact format;
2. an **output parser** converts the text into Python objects (`JsonOutputParser` → `dict`).

> **Curly braces are special in templates.** `{x}` is a variable; a literal brace must be written `{{` or `}}`. That is why the JSON keys below are described in words instead of with a literal `{...}` example.
>
> Tip: `ChatOllama(..., format="json")` forces the model to emit valid JSON.

In [19]:
#%% 1.5 JSON output with a parser
from langchain_core.output_parsers import JsonOutputParser

glossary_prompt = ChatPromptTemplate.from_messages([
    ("system", "You write glossary entries for a programming course. "
               "Return ONLY a JSON object with the keys \"term\", \"definition\" and \"example\"."),
    ("human", "Term: {term}"),
])
glossary_chain = glossary_prompt | llm | JsonOutputParser()

entry = glossary_chain.invoke({"term": "list comprehension"})
print(type(entry).__name__, "→", entry)

# Literal braces must be doubled: only {q} is a variable here
print(ChatPromptTemplate.from_template('Answer as {{"answer": "..."}}. Question: {q}').input_variables)

dict → {'term': 'list comprehension', 'definition': 'A concise way to create lists in Python by iterating over a sequence and applying a transformation to each element.', 'example': 'numbers = [1, 2, 3, 4, 5]; squared_numbers = [n**2 for n in numbers]; print(squared_numbers)'}
['q']


### 1.6 `MessagesPlaceholder`: a slot for a whole list of messages
An LLM is **stateless**: it only knows what is in the current prompt. To make it "remember" a conversation, we put the previous messages back into the prompt. `MessagesPlaceholder("history")` is a slot that receives a whole list of messages. In a later lab (*chains with memory*) LangChain will fill this slot automatically.

In [20]:
#%% 1.6 MessagesPlaceholder
from langchain_core.prompts import MessagesPlaceholder
from langchain_core.messages import HumanMessage, AIMessage

chat_prompt = ChatPromptTemplate.from_messages([
    ("system", "You are a helpful programming tutor. Be brief."),
    MessagesPlaceholder("history"),
    ("human", "{question}"),
])
history = [
    HumanMessage("My name is Ana and I am learning Python."),
    AIMessage("Nice to meet you, Ana! Python is a great first language."),
]
inputs = {"history": history, "question": "What is my name and what am I learning?"}
for m in chat_prompt.invoke(inputs).to_messages():
    m.pretty_print()
print("\nAnswer:", (chat_prompt | llm | StrOutputParser()).invoke(inputs))
print("Without history:", (chat_prompt | llm | StrOutputParser()).invoke({**inputs, "history": []}))

================================ System Message ================================

You are a helpful programming tutor. Be brief.
================================ Human Message =================================

My name is Ana and I am learning Python.
================================== Ai Message ==================================

Nice to meet you, Ana! Python is a great first language.
================================ Human Message =================================

What is my name and what am I learning?

Answer: Your name is Ana, and you're learning Python.
Without history: I don't have any information about you, so I don't know your name. However, I can tell you that you're learning programming concepts and seeking guidance from me, a programming tutor. How can I assist you today?


### ✏️ Exercise 1 — From f-string to template
The cell below is a code reviewer written "the old way". Rewrite it with a `ChatPromptTemplate`:
1. A **system** message with the reviewer's role, a limit of `{max_issues}` issues, and the output format: a numbered list where each item is *construct → problem → fix*. Also say what to answer when the code has no real problems.
2. A **human** message with `{language}` and `{code}`; put the code between triple backticks (delimiters separate data from instructions).
3. Print `input_variables`, then use `.partial()` to fix `max_issues=3` and print them again.
4. Build `review_chain = ... | llm | StrOutputParser()` and run it with **one** `.batch()` call over the three `SNIPPETS`.
5. Compare with the old version: which output is easier to read and to use in code?

In [21]:
#%% Exercise 1 — the "old way" (do not modify)
def review_old_style(code, language="Python"):
    text = ("You are a strict code reviewer. Review this " + language + " code and list at most 3 problems "
            "with a fix for each:\n" + code)
    return llm.invoke(text).content

SNIPPETS = [
    "def average(xs):\n    return sum(xs) / len(xs)",
    "for i in range(len(names)):\n    print(names[i])",
    "def add_item(item, items=[]):\n    items.append(item)\n    return items",
]
print(review_old_style(SNIPPETS[2]))

**Code Review**

The provided Python code has several issues that need to be addressed. Here are three problems with suggested fixes:

### Problem 1: Mutable Default Argument

The `items` parameter has a mutable default value, which can lead to unexpected behavior. In Python, default argument values are evaluated only once at the point of function definition in the defining scope. This means that if you use a mutable default argument and mutate it, you will and have mutated that object for all future calls to the function as well.

**Fix:**
```python
def add_item(item, items=None):
    if items is None:
        items = []
    items.append(item)
    return items
```
By setting `items` to `None` by default, we ensure that a new list is created each time the function is called, rather than reusing the same list.

### Problem 2: Lack of Input Validation

The function does not validate its input. If `item` is not a valid item (e.g., a string, integer, etc.), it may cause an error when tryin

In [22]:
#%% Exercise 1 — your solution
# review_prompt = ChatPromptTemplate.from_messages([
#     ("system", "..."),
#     ("human", "..."),
# ])
# print("before partial:", review_prompt.input_variables)
# review_prompt_3 = review_prompt.partial(...)
# print("after partial :", review_prompt_3.input_variables)
#
# review_chain = ...
# reviews = review_chain.batch([...])
# for s, r in zip(SNIPPETS, reviews):
#     print("━━━ CODE ━━━\n" + s + "\n━━━ REVIEW ━━━\n" + r + "\n")

---
## Part 2 — Improving prompts with the LangChain Hub

The **LangChain Hub** (part of LangSmith) is a public registry of prompts, identified as `owner/name` — or `owner/name:commit` to **pin** an exact version. Browse it at <https://smith.langchain.com/hub>. It is useful to:
- start from prompts that other people have already tested,
- share prompts inside a team,
- version prompts **separately from the code** (change the prompt without redeploying).

Prompts are pulled with the LangSmith client, `Client().pull_prompt("owner/name")`. (Older tutorials use `from langchain import hub; hub.pull(...)`: same idea.)

### Access
Pulling may require a **free LangSmith API key**:
1. Create an account at <https://smith.langchain.com> → *Settings → API Keys → Create API Key*.
2. In Colab: 🔑 **Secrets** → *Add new secret* → name `LANGSMITH_API_KEY` → switch on **Notebook access**.

No key or no internet? Every cell in this part falls back to a **local copy** of the prompt, so you can still complete the lab.

In [23]:
#%% 2.1 (Optional) LangSmith API key from Colab Secrets
os.environ.setdefault("LANGSMITH_TRACING", "false")      # we only use the Hub, not tracing
try:
    from google.colab import userdata
    os.environ["LANGSMITH_API_KEY"] = userdata.get("LANGSMITH_API_KEY")
    print("✅ LANGSMITH_API_KEY loaded from Colab Secrets.")
except Exception:
    print("ℹ️ No LANGSMITH_API_KEY found — we will try anyway and use local copies if needed.")

ℹ️ No LANGSMITH_API_KEY found — we will try anyway and use local copies if needed.


In [24]:
#%% 2.2 A helper to pull prompts with a local fallback
from langsmith import Client

def pull_prompt(name, fallback, expected_vars=None):
    # Pull `name` from the Hub; return `fallback` if that fails or the input variables are not the expected ones.
    try:
        hub_prompt = Client().pull_prompt(name)
        if expected_vars and set(hub_prompt.input_variables) != set(expected_vars):
            raise ValueError(f"unexpected input variables {hub_prompt.input_variables}")
        print(f"✅ Pulled '{name}' from the Hub")
        return hub_prompt
    except Exception as e:
        print(f"⚠️ Could not use '{name}' from the Hub ({type(e).__name__}: {str(e)[:100]}) → local fallback")
        return fallback

def show_prompt(p):
    # Print each message template of a ChatPromptTemplate.
    for m in p.messages:
        text = getattr(getattr(m, "prompt", None), "template", m)
        print(f"\n[{type(m).__name__}]\n{text}")

### 2.3 Pull and inspect a popular prompt: `rlm/rag-prompt`
`rlm/rag-prompt` is one of the most downloaded prompts on the Hub. It answers a question **using only a given context** — the core of Retrieval-Augmented Generation (RAG), which we will study later. Before using any third-party prompt, **read it**: what variables does it need, what does it tell the model to do?

In [25]:
#%% 2.3 Pull, inspect and use rlm/rag-prompt
RAG_FALLBACK = ChatPromptTemplate.from_messages([
    ("human", "You answer questions using only the context below. If the context does not contain the answer, "
              "say that you don't know. Use at most three sentences.\n\n"
              "Question: {question}\nContext: {context}\nAnswer:"),
])
rag_prompt = pull_prompt("rlm/rag-prompt", RAG_FALLBACK, expected_vars=["context", "question"])
print("input variables:", rag_prompt.input_variables)
show_prompt(rag_prompt)

CONTEXT = ("Lab submissions for the AI Agents course are due on Sundays at 23:59 (Ecuador time). "
           "Late submissions lose 10% per day and are not accepted after 3 days.")
rag_chain = rag_prompt | llm | StrOutputParser()
print()
for q in ["When are lab submissions due?", "What happens if I submit 2 days late?", "Who is the course instructor?"]:
    print(f"Q: {q}\nA: {rag_chain.invoke({'context': CONTEXT, 'question': q})}\n")

⚠️ Could not use 'rlm/rag-prompt' from the Hub (ValueError: Pulling a public prompt by owner/name is disabled by default because prompts may contain untrusted s) → local fallback
input variables: ['context', 'question']

[HumanMessagePromptTemplate]
You answer questions using only the context below. If the context does not contain the answer, say that you don't know. Use at most three sentences.

Question: {question}
Context: {context}
Answer:

Q: When are lab submissions due?
A: Lab submissions for the AI Agents course are due on Sundays at 23:59 (Ecuador time).

Q: What happens if I submit 2 days late?
A: If you submit 2 days late, you will lose 10% of the total penalty for each day, and your submission will not be accepted after 3 days. The exact amount of the penalty will depend on the number of days late, but it will be 10% per day.

Q: Who is the course instructor?
A: I don't know. The context only provides information about the course, but it does not mention the instructor.



### 2.4 A prompt that writes prompts
Writing a good prompt is itself a task an LLM can help with (**meta-prompting**). The Hub prompt `hardkothari/prompt-maker` takes a **task description** and a **lazy prompt** and returns a detailed, improved prompt. We use it on a one-line prompt.

In [26]:
#%% 2.4 Improve a lazy prompt with prompt-maker
MAKER_FALLBACK = ChatPromptTemplate.from_messages([
    ("system", "You are an expert prompt engineer. Rewrite the user's short, vague prompt into a clear and complete "
               "prompt for a large language model. The improved prompt must state the role the model should take, "
               "the task, the audience, the constraints (length, tone, what to avoid) and the exact output format. "
               "Return only the improved prompt, written in the second person (\"You are...\")."),
    ("human", "Task: {task}\nLazy prompt: {lazy_prompt}"),
])
maker_prompt = pull_prompt("hardkothari/prompt-maker", MAKER_FALLBACK, expected_vars=["task", "lazy_prompt"])
maker_chain = maker_prompt | llm_long | StrOutputParser()

TASK = "Summarize the abstract of a research paper for first-year computer science students."
LAZY_PROMPT = "Summarize this abstract."
improved_prompt_text = maker_chain.invoke({"task": TASK, "lazy_prompt": LAZY_PROMPT})
print(improved_prompt_text)

⚠️ Could not use 'hardkothari/prompt-maker' from the Hub (ValueError: Pulling a public prompt by owner/name is disabled by default because prompts may contain untrusted s) → local fallback
You are to summarize the abstract of a research paper, written in a clear and concise manner suitable for first-year computer science students, approximately 150-200 words in length, avoiding technical jargon and overly complex concepts, and presenting the main findings and implications of the research in a way that is accessible and engaging for an audience with a basic understanding of computer science fundamentals.


### 2.5 Use the improved prompt and compare
The improved prompt is **text written by a model**: it may contain `{` `}` (for example a JSON format), which a template would read as variables. The helper `as_literal` escapes them before we turn the text into a system message.

In [27]:
#%% 2.5 Lazy vs improved prompt on the same abstract
def as_literal(text):
    # Escape braces so that model-written text is not read as template variables.
    return text.replace("{", "{{").replace("}", "}}")

lazy_template = ChatPromptTemplate.from_messages([("human", LAZY_PROMPT + "\n\n{abstract}")])
improved_template = ChatPromptTemplate.from_messages([
    ("system", as_literal(improved_prompt_text)),
    ("human", "Abstract:\n{abstract}"),
])

ABSTRACT = (
    "Large language models can be turned into autonomous agents by letting them alternate between "
    "reasoning steps and actions such as web searches or calculator calls. We study this design on three "
    "question-answering benchmarks and two interactive decision-making environments. Interleaving reasoning "
    "with actions reduces factual errors compared with reasoning alone, because the model can check facts "
    "against external sources, and it improves success rates compared with acting alone, because the model can "
    "plan and recover from mistakes. We also find that the resulting trajectories are easier for humans to "
    "inspect, although the approach remains sensitive to the quality of the retrieved information."
)
for name, tpl in [("LAZY", lazy_template), ("IMPROVED", improved_template)]:
    print(f"━━━ {name} prompt ━━━\n{(tpl | llm_long | StrOutputParser()).invoke({'abstract': ABSTRACT})}\n")

━━━ LAZY prompt ━━━
This abstract describes a design where large language models are used as autonomous agents by alternating between reasoning steps and taking actions (e.g. web searches). The study found that this approach reduces factual errors and improves success rates compared to relying solely on reasoning or actions. However, the quality of the retrieved information still affects the results.

━━━ IMPROVED prompt ━━━
Here's a summary of the abstract in 150-200 words:

Imagine a computer program that can think and act like a human. Researchers have been working on creating a new type of AI that can alternate between thinking and taking actions, like searching the web or solving math problems. They tested this approach on three different tasks that require answering questions and two tasks that require making decisions.

The results show that this new approach is more accurate than just thinking alone, because the AI can check its answers against external sources. It also perform

### 2.6 What makes a prompt "better"?
Use this checklist to read the improved prompt (and every prompt you write):

| Element | Question it answers | Example |
|---|---|---|
| **Role** | Who should the model be? | "You are a patient tutor for first-year students" |
| **Task** | What exactly must it do? | "Summarize the abstract" |
| **Context / audience** | For whom, with what background? | "Readers know basic programming, no ML" |
| **Constraints** | Length, tone, what to avoid | "At most 80 words, no jargon" |
| **Output format** | What shape must the answer have? | "Three bullet points: problem, method, result" |
| **Delimiters** | Where do instructions end and data begin? | text between ``` or under a heading |
| **Examples** | What does a good answer look like? | → Part 3 |

👉 **Try it:** tick the elements present in `improved_prompt_text`. Which is missing?

**Practical notes on the Hub**
- **Pin versions** (`owner/name:commit`) in real projects: a prompt that changes under you is a silent bug.
- Many Hub prompts were written for large commercial models; a long, complex prompt may be **too much for a 3B model**. Test, then simplify.
- A third-party prompt is **untrusted text** that you send to your model: read it as you would read third-party code.
- With an API key you can publish your own prompts: `Client().push_prompt("my-prompt", object=my_template)`.

### ✏️ Exercise 2 — Improve a prompt from your own work
1. Choose a task from your field (or: *give feedback on a student's answer*, *write a quiz question about a topic*, *explain a Python error message*). Write it in `MY_TASK`, and a **one-line** lazy prompt in `MY_LAZY_PROMPT`.
2. Improve it with `maker_chain`.
3. **Edit it by hand**: add what the checklist says is missing, and shorten it if it is too long for a small model. Save it in `MY_EDITED_PROMPT`.
4. Run the three versions (lazy, prompt-maker, edited) on **3 inputs** with `.batch()`.
5. Score every output 0–2 on three criteria — **format** followed, **correct**, right for the **audience** — and show the totals in a table. Which version won, and was the prompt-maker version actually better than your edit?

In [28]:
#%% Exercise 2 — your solution
MY_TASK = ""            # e.g. "Explain a Python error message to a first-year student and how to fix it."
MY_LAZY_PROMPT = ""     # e.g. "Explain this error."
MY_INPUTS = [
    # "input 1", "input 2", "input 3",
]

# maker_version = maker_chain.invoke({"task": MY_TASK, "lazy_prompt": MY_LAZY_PROMPT})
# print(maker_version)

MY_EDITED_PROMPT = ""   # your manual edit

# versions = {
#     "lazy":         ChatPromptTemplate.from_messages([("human", MY_LAZY_PROMPT + "\n\n{x}")]),
#     "prompt-maker": ChatPromptTemplate.from_messages([("system", as_literal(maker_version)), ("human", "{x}")]),
#     "edited":       ChatPromptTemplate.from_messages([("system", MY_EDITED_PROMPT), ("human", "{x}")]),
# }
# results = {name: (tpl | llm_long | StrOutputParser()).batch([{"x": x} for x in MY_INPUTS])
#            for name, tpl in versions.items()}

# SCORES = {"lazy": [...], "prompt-maker": [...], "edited": [...]}   # 0–2 per criterion and input

---
## Part 3 — Few-shot prompting

A model can learn a task **from examples placed in the prompt**, without changing any weight. This is called **in-context learning** (Brown et al., 2020):
- **zero-shot**: only instructions;
- **one-shot / few-shot**: instructions + one or a few *input → output* demonstrations.

Examples are especially good at teaching what is hard to describe in words: the **output format**, the **label space**, the **style**, and how to handle **borderline cases**. With chat models, the best way to write an example is as a fake conversation turn: a `human` message with the input followed by an `ai` message with the ideal answer.

### Our task: routing messages in a course forum
An assistant for a course must route every forum message to the right person. We **measure** each prompt on a fixed test set with two numbers:
- **accuracy** — is the predicted category right?
- **exact-format rate** — did the model answer with *only* the category name? (An agent's code needs a parseable answer.)

In [29]:
#%% 3.1 Labels and test set
import textwrap
import pandas as pd

LABELS = ["deadline", "grading", "concept_doubt", "technical_issue", "other"]
LABEL_GUIDE = (
    "deadline: due dates, extensions, late submissions\n"
    "grading: scores, rubrics, regrade requests, how the grade is computed\n"
    "concept_doubt: questions about course content or how something works\n"
    "technical_issue: problems with Colab, installation, the course platform or the autograder\n"
    "other: anything else"
)
TEST_SET = [
    ("Can I submit lab 4 on Monday instead of Sunday? I was sick.", "deadline"),
    ("Why did I get 6/10 in midterm question 3? I think my answer was right.", "grading"),
    ("What is the difference between a list and a tuple?", "concept_doubt"),
    ("Colab says 'CUDA out of memory' when I load the model.", "technical_issue"),
    ("Is there a study group for this course?", "other"),
    ("¿Hasta cuándo se puede entregar el laboratorio 3?", "deadline"),
    ("The autograder shows 0 points but my code runs fine on my laptop.", "technical_issue"),
    ("How is the final grade computed? What percentage are the labs?", "grading"),
    ("I don't understand why the REINFORCE gradient uses the log of the policy.", "concept_doubt"),
    ("Thanks for the great lecture today!", "other"),
    ("pip install fails with 'No matching distribution found for langchain-ollama'.", "technical_issue"),
    ("Will the quiz deadline be extended because of the holiday?", "deadline"),
]
pd.DataFrame(TEST_SET, columns=["message", "gold label"])

,message,gold label
0,Can I submit lab 4 on Monday instead of Sunday...,deadline
1,Why did I get 6/10 in midterm question 3? I th...,grading
2,What is the difference between a list and a tu...,concept_doubt
3,Colab says 'CUDA out of memory' when I load th...,technical_issue
4,Is there a study group for this course?,other
5,¿Hasta cuándo se puede entregar el laboratorio 3?,deadline
6,The autograder shows 0 points but my code runs...,technical_issue
7,How is the final grade computed? What percenta...,grading
8,I don't understand why the REINFORCE gradient ...,concept_doubt
9,Thanks for the great lecture today!,other


### 3.2 Zero-shot baseline
The system message explains the categories; the human message is the forum message. `parse_label` maps the raw answer to a label (or `INVALID`), and `evaluate` runs a chain over the whole test set with one `.batch()` call.

In [30]:
#%% 3.2 Zero-shot baseline
SYSTEM_CLASSIFIER = ("You classify messages from a university course forum into exactly one category.\n"
                     "Categories:\n" + LABEL_GUIDE + "\n"
                     "Answer with the category name only.")

zero_shot_prompt = ChatPromptTemplate.from_messages([("system", SYSTEM_CLASSIFIER), ("human", "{input}")])
zero_shot_chain = zero_shot_prompt | llm | StrOutputParser()

def parse_label(text):
    # Map the raw model output to the first label it mentions, or 'INVALID'.
    t = text.strip().lower().replace(" ", "_")
    hits = [(t.find(label), label) for label in LABELS if label in t]
    return min(hits)[1] if hits else "INVALID"

def evaluate(chain, data, name):
    # Run `chain` on every message of `data` and return a DataFrame with predictions.
    outputs = chain.batch([{"input": msg} for msg, _ in data], config={"max_concurrency": 4})
    rows = []
    for (msg, gold), out in zip(data, outputs):
        pred = parse_label(out)
        rows.append({"message": textwrap.shorten(msg, 55), "gold": gold,
                     "raw output": textwrap.shorten(out.strip(), 40), "pred": pred,
                     "exact_format": out.strip().strip(".").lower() in LABELS, "correct": pred == gold})
    df = pd.DataFrame(rows)
    print(f"{name:32s} accuracy = {df.correct.mean():4.0%} | exact-format rate = {df.exact_format.mean():4.0%}")
    return df

zero_df = evaluate(zero_shot_chain, TEST_SET, "zero-shot")
display(zero_df)

zero-shot                        accuracy =  92% | exact-format rate = 100%


,message,gold,raw output,pred,exact_format,correct
0,Can I submit lab 4 on Monday instead of Sunday...,deadline,deadline,deadline,True,True
1,Why did I get 6/10 in midterm question 3? I th...,grading,concept_doubt,concept_doubt,True,False
2,What is the difference between a list and a tu...,concept_doubt,concept_doubt,concept_doubt,True,True
3,Colab says 'CUDA out of memory' when I load th...,technical_issue,technical_issue,technical_issue,True,True
4,Is there a study group for this course?,other,other,other,True,True
5,¿Hasta cuándo se puede entregar el laboratorio 3?,deadline,deadline,deadline,True,True
6,The autograder shows 0 points but my code runs...,technical_issue,technical_issue,technical_issue,True,True
7,How is the final grade computed? What percenta...,grading,grading,grading,True,True
8,I don't understand why the REINFORCE gradient ...,concept_doubt,concept_doubt,concept_doubt,True,True
9,Thanks for the great lecture today!,other,other,other,True,True


### 3.3 Few-shot with `FewShotChatMessagePromptTemplate`
We need three pieces:
1. a list of examples, each a `dict` (`{"input": ..., "output": ...}`);
2. an `example_prompt` that formats **one** example as a `human` → `ai` pair;
3. a `FewShotChatMessagePromptTemplate`, inserted between the system message and the real question.

The system message is **the same** as in the zero-shot prompt, so any difference in the results comes from the examples alone (a fair comparison). None of the examples appears in the test set: putting test items in the prompt would be *data leakage*.

In [31]:
#%% 3.3 Few-shot prompt with 10 fixed examples (2 per label)
from langchain_core.prompts import FewShotChatMessagePromptTemplate

EXAMPLES = [
    {"input": "Is the project due this Friday or next Friday?", "output": "deadline"},
    {"input": "I missed the deadline because my internet was down. Can I still send it?", "output": "deadline"},
    {"input": "Could you check my lab 2 grade again? The rubric says 2 points for the plot.", "output": "grading"},
    {"input": "Do quizzes count for the final grade?", "output": "grading"},
    {"input": "Why do we need a discount factor in reinforcement learning?", "output": "concept_doubt"},
    {"input": "What does the temperature parameter do exactly?", "output": "concept_doubt"},
    {"input": "My notebook crashes when I run the install cell.", "output": "technical_issue"},
    {"input": "I can't log in to the course platform to upload my file.", "output": "technical_issue"},
    {"input": "Where is the classroom for tomorrow's session?", "output": "other"},
    {"input": "Can you recommend a good book about AI agents?", "output": "other"},
]
example_prompt = ChatPromptTemplate.from_messages([("human", "{input}"), ("ai", "{output}")])

def make_few_shot_chain(examples, system=SYSTEM_CLASSIFIER):
    # Build system message + examples (as human/ai turns) + the real input; return (prompt, chain).
    few_shot = FewShotChatMessagePromptTemplate(example_prompt=example_prompt, examples=examples)
    prompt = ChatPromptTemplate.from_messages([("system", system), few_shot, ("human", "{input}")])
    return prompt, prompt | llm | StrOutputParser()

few_shot_prompt, few_shot_chain = make_few_shot_chain(EXAMPLES)

msgs = few_shot_prompt.invoke({"input": "Hello everyone!"}).to_messages()
print(f"The prompt now contains {len(msgs)} messages. The first five and the last one:")
for m in msgs[:5]:
    m.pretty_print()
print("   ...")
msgs[-1].pretty_print()

The prompt now contains 22 messages. The first five and the last one:
================================ System Message ================================

You classify messages from a university course forum into exactly one category.
Categories:
deadline: due dates, extensions, late submissions
grading: scores, rubrics, regrade requests, how the grade is computed
concept_doubt: questions about course content or how something works
technical_issue: problems with Colab, installation, the course platform or the autograder
other: anything else
Answer with the category name only.
================================ Human Message =================================

Is the project due this Friday or next Friday?
================================== Ai Message ==================================

deadline
================================ Human Message =================================

I missed the deadline because my internet was down. Can I still send it?
================================== Ai Message

In [32]:
#%% 3.4 Evaluate and compare
few_df = evaluate(few_shot_chain, TEST_SET, "few-shot (10 fixed examples)")
display(few_df)

def prompt_tokens(prompt):
    # Number of input tokens the model receives for one test message (the cost of the prompt).
    usage = (prompt | llm).invoke({"input": TEST_SET[0][0]}).usage_metadata or {}
    return usage.get("input_tokens")

summary = pd.DataFrame({
    "zero-shot": {"accuracy": zero_df.correct.mean(), "exact format": zero_df.exact_format.mean(),
                  "input tokens": prompt_tokens(zero_shot_prompt)},
    "few-shot (10 fixed)": {"accuracy": few_df.correct.mean(), "exact format": few_df.exact_format.mean(),
                            "input tokens": prompt_tokens(few_shot_prompt)},
}).T
display(summary)

few-shot (10 fixed examples)     accuracy = 100% | exact-format rate = 100%


,message,gold,raw output,pred,exact_format,correct
0,Can I submit lab 4 on Monday instead of Sunday...,deadline,deadline,deadline,True,True
1,Why did I get 6/10 in midterm question 3? I th...,grading,grading,grading,True,True
2,What is the difference between a list and a tu...,concept_doubt,concept_doubt,concept_doubt,True,True
3,Colab says 'CUDA out of memory' when I load th...,technical_issue,technical_issue,technical_issue,True,True
4,Is there a study group for this course?,other,other,other,True,True
5,¿Hasta cuándo se puede entregar el laboratorio 3?,deadline,deadline,deadline,True,True
6,The autograder shows 0 points but my code runs...,technical_issue,technical_issue,technical_issue,True,True
7,How is the final grade computed? What percenta...,grading,grading,grading,True,True
8,I don't understand why the REINFORCE gradient ...,concept_doubt,concept_doubt,concept_doubt,True,True
9,Thanks for the great lecture today!,other,other,other,True,True


,accuracy,exact format,input tokens
zero-shot,0.916667,1.0,128.0
few-shot (10 fixed),1.000000,1.0,363.0


### 3.5 Choosing the examples dynamically
With a large pool of examples we cannot put all of them in the prompt (cost, context window). An **example selector** picks, for every new input, the examples **most similar** to it (Liu et al., 2022):
1. every example input is converted into a vector by an **embedding model** and stored in a vector store;
2. the new input is embedded too, and the `k` nearest examples are inserted into the prompt.

We use `nomic-embed-text` through Ollama and LangChain's simple `InMemoryVectorStore`.

In [33]:
#%% 3.5 Semantic similarity example selector
from langchain_core.example_selectors import SemanticSimilarityExampleSelector
from langchain_core.vectorstores import InMemoryVectorStore
from langchain_ollama import OllamaEmbeddings

embeddings = OllamaEmbeddings(model=EMBED_MODEL)
selector = SemanticSimilarityExampleSelector.from_examples(
    EXAMPLES, embeddings, InMemoryVectorStore, k=3, input_keys=["input"])

for query in ["The autograder gives me zero points but everything works locally.",
              "When do we have to hand in the final project?"]:
    print(f"\nQuery: {query}")
    for ex in selector.select_examples({"input": query}):
        print(f"   {ex['output']:16s} ← {ex['input']}")

dyn_few_shot = FewShotChatMessagePromptTemplate(example_prompt=example_prompt, example_selector=selector,
                                                input_variables=["input"])
dyn_prompt = ChatPromptTemplate.from_messages([("system", SYSTEM_CLASSIFIER), dyn_few_shot, ("human", "{input}")])
dyn_chain = dyn_prompt | llm | StrOutputParser()

print()
dyn_df = evaluate(dyn_chain, TEST_SET, "few-shot (3 similar examples)")
summary.loc["few-shot (3 similar)"] = {"accuracy": dyn_df.correct.mean(), "exact format": dyn_df.exact_format.mean(),
                                       "input tokens": prompt_tokens(dyn_prompt)}
display(summary)


Query: The autograder gives me zero points but everything works locally.
   grading          ← Could you check my lab 2 grade again? The rubric says 2 points for the plot.
   technical_issue  ← My notebook crashes when I run the install cell.
   concept_doubt    ← What does the temperature parameter do exactly?

Query: When do we have to hand in the final project?
   deadline         ← Is the project due this Friday or next Friday?
   deadline         ← I missed the deadline because my internet was down. Can I still send it?
   grading          ← Do quizzes count for the final grade?

few-shot (3 similar examples)    accuracy =  92% | exact-format rate = 100%


,accuracy,exact format,input tokens
zero-shot,0.916667,1.0,128.0
few-shot (10 fixed),1.000000,1.0,363.0
few-shot (3 similar),0.916667,1.0,208.0


👉 **Read the tables, not only the totals.** Which test messages changed from wrong to right (or right to wrong) between the three prompts? With only 12 test messages, one message is worth ≈ 8 percentage points: be careful with conclusions.

### ✏️ Exercise 3 — Experiments with examples
**(a) How many examples?** Write `examples_per_label(n)`, which returns the first `n` examples of **each** label (`n = 0` → empty list, which is the zero-shot prompt). Evaluate `n = 0, 1, 2` and show accuracy, exact-format rate and input tokens in one table.

**(b) Majority and recency bias.** Build an **unbalanced** example list: one example of each label except `technical_issue`, followed by 6 `technical_issue` examples (so that label is both the majority and the last one the model sees). Evaluate it and print `df.pred.value_counts()`. Does the model predict `technical_issue` more often than it should?

**(c) Fix a failure with one example.** Pick a message that the 10-example prompt got wrong (if none, pick the one you find most borderline). Write **one or two new examples** that teach the rule behind it — *not* a copy of the test message — add them to `EXAMPLES`, and re-evaluate. Did it fix the error without breaking other messages?

In [34]:
#%% Exercise 3a — number of examples per label
def examples_per_label(n):
    # TODO: return the first n examples of EACH label in LABELS
    ...

# rows = []
# for n in (0, 1, 2):
#     prompt, chain = make_few_shot_chain(examples_per_label(n))
#     df = evaluate(chain, TEST_SET, f"{n} example(s) per label")
#     rows.append({"examples per label": n, "accuracy": ..., "exact format": ..., "input tokens": ...})
# display(pd.DataFrame(rows).set_index("examples per label"))

In [35]:
#%% Exercise 3b — unbalanced examples
# extra_tech = [ ... 4 new technical_issue examples ... ]
# one_each = [ ... one example of each label except technical_issue ... ]
# unbalanced = one_each + [e for e in EXAMPLES if e["output"] == "technical_issue"] + extra_tech
# _, unb_chain = make_few_shot_chain(unbalanced)
# unb_df = evaluate(unb_chain, TEST_SET, "unbalanced (6 technical_issue last)")
# print(unb_df.pred.value_counts())

In [36]:
#%% Exercise 3c — fix a failure
# display(few_df[~few_df.correct])
# EXTRA = [{"input": "...", "output": "..."}]
# _, fixed_chain = make_few_shot_chain(EXAMPLES + EXTRA)
# fixed_df = evaluate(fixed_chain, TEST_SET, "few-shot + targeted examples")

---
## Part 4 — Questions
Answer briefly (2–4 sentences each), using your outputs and tables as evidence.

**Q1.** Give three concrete advantages of a `ChatPromptTemplate` over an f-string prompt. Use at least one thing you **observed** in Part 1 (e.g. cell 1.1 or Exercise 1).

**Q2.** When is `.partial()` useful? Explain the difference between a partial **value** and a partial **function**, with an example of each from this lab.

**Q3.** List two benefits and two risks of pulling prompts from the LangChain Hub. Why did we escape the braces of the prompt-maker's output before using it as a template?

**Q4.** Compare zero-shot and few-shot (10 fixed examples) in cell 3.4. Which metric improved more — accuracy or exact-format rate — and why do you think examples help that metric in particular? What did you pay for the improvement?

**Q5.** Compare fixed examples with semantically selected examples (3.5): accuracy, cost and possible risks. Relate your answer to what you saw in Exercise 3b.

**Q6.** Why did we use `temperature=0` for all the evaluations (remember Lab 2.5)? What would you have to change in `evaluate` if the model were sampled at `temperature=1`? Finally: our examples only show the **answer**, not **how** to reach it. For what kind of task is that a problem?

**Your answers:**

- **Q1.**
- **Q2.**
- **Q3.**
- **Q4.**
- **Q5.**
- **Q6.**

---
### What's next
In the next lab we keep the same tools (`ChatPromptTemplate`, few-shot examples, `batch`) and use them for **reasoning**: zero-shot and few-shot **Chain-of-Thought**, **self-consistency** (many samples + majority vote) and **self-feedback** (the model critiques and improves its own answer).

### References
- T. Brown et al. *Language Models are Few-Shot Learners.* NeurIPS 2020 (in-context learning).
- Z. Zhao, E. Wallace, S. Feng, D. Klein, S. Singh. *Calibrate Before Use: Improving Few-Shot Performance of Language Models.* ICML 2021 (majority-label and recency bias).
- J. Liu et al. *What Makes Good In-Context Examples for GPT-3?* DeeLIO 2022 (similarity-based example selection).
- S. Min et al. *Rethinking the Role of Demonstrations: What Makes In-Context Learning Work?* EMNLP 2022.
- LangChain documentation (prompt templates, few-shot prompting, LCEL): <https://docs.langchain.com>
- LangSmith Prompt Hub: <https://smith.langchain.com/hub>
- Ollama and its model library: <https://ollama.com> · <https://ollama.com/library>